# How much should score questions respect level order? (`ordinal_weight`)

Score questions ("how severe, 0 to 3") have ordered levels. Cross-entropy alone treats them as
unrelated categories: predicting 2 when the answer is 3 costs as much as predicting 0. sysone adds
a ranked-probability term for score questions, weighted by `ordinal_weight` (default 1.0; 0 turns
it off), so near misses cost less than far ones.

This notebook measures that term on
[LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) with
ModernBERT-base, to answer three questions:

1. **Does it help?** `score_error`, the average gap between the predicted and the right level,
   should drop when `ordinal_weight` is above 0.
2. **Does it cost anything?** Exact-level score accuracy shouldn't fall, and choice and noul
   accuracy should move no more than noise: the term only adds to score questions' loss.
3. **How strong should it be?** Whether the default of 1.0 is right, and whether different score
   questions want different weights.

> **Cost.** Step 1 trains only the decision head, on backbone features computed once and shared by
> every weight, so it takes minutes. Step 2 fine-tunes the whole backbone twice, which is the
> expensive part: use a CUDA GPU (on Colab, an L4 or A100 runtime), and check the `minutes` column
> of the first run before starting the second.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it.
# ordinal_weight needs sysone 0.3; until it is on PyPI, this installs the development branch.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone @ git+https://github.com/ForwardMetric/sysone@milestone-3-calibration"

In [ ]:
import json
import time

import pandas as pd
from datasets import load_dataset

from sysone import EncoderModel

WEIGHTS = [0, 0.5, 1, 2]           # ordinal_weight values for the head-only sweep
FULL_EPOCHS = 8                    # upper bound for the full-tuning runs; early stopping may end them sooner
try:  # on Colab, keep the feature cache on the runtime's disk
    import google.colab  # noqa: F401
    CACHE_DIR = "/content/feature-cache"
except ImportError:
    CACHE_DIR = "feature-cache"

## Load the data

The same records as `encoder_typed_decisions.ipynb`: soft labels from several raters, with the
workflow as the category. Each workflow has score questions among its five.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))

## What each run reports

`summarize` scores a trained model on the test split: accuracy overall and per question type,
`score_error` and exact-level `score_accuracy` for score questions, and how long the run took.
`score_error_by_question` keeps each score question's error, for question 3.

In [ ]:
def summarize(name, weight, model, started):
    report = model.evaluate(test)
    score = report["by_type"]["score"]
    return {
        "run": name,
        "ordinal_weight": weight,
        "minutes": round((time.time() - started) / 60, 1),
        "accuracy": report["overall"]["accuracy"],
        "choice_accuracy": report["by_type"]["choice"]["accuracy"],
        "noul_accuracy": report["by_type"]["noul"]["accuracy"],
        "score_accuracy": score["accuracy"],
        "score_error": score["score_error"],
        "score_error_by_question": {q: m["score_error"] for q, m in report["by_question"].items()
                                    if "score_error" in m},
    }


def table(rows):
    columns = ["ordinal_weight", "minutes", "accuracy", "choice_accuracy", "noul_accuracy", "score_accuracy",
               "score_error"]
    return pd.DataFrame(rows).set_index("run")[columns].round(4)


def by_question(rows):
    """Score error per score question (rows) for each run (columns)."""
    return pd.DataFrame({r["run"]: r["score_error_by_question"] for r in rows}).round(4)

## Step 1: a quick sweep, training only the head

With `tuning="head"` the backbone is frozen, so its features are computed once, written to
`CACHE_DIR`, and reused by every weight; each run then trains only the small decision head. This
is a cheap first signal, not the final answer: a frozen ModernBERT reads these records poorly, so
effects can be smaller here than with full tuning.

In [ ]:
head_rows = []
for weight in WEIGHTS:
    started = time.time()
    model = EncoderModel(tuning="head", ordinal_weight=weight, cache_dir=CACHE_DIR, verbose=0).fit(train)
    head_rows.append(summarize("head, ordinal_weight=%g" % weight, weight, model, started))
table(head_rows)

In [ ]:
by_question(head_rows)

The weight to confirm in step 2 is the one with the lowest `score_error` among those above 0, as
long as its `score_accuracy` is within a point of `ordinal_weight=0`. Override `BEST` by hand if the
table says otherwise.

In [ ]:
baseline = next(r for r in head_rows if r["ordinal_weight"] == 0)
candidates = [r for r in head_rows if r["ordinal_weight"] > 0
              and r["score_accuracy"] >= baseline["score_accuracy"] - 0.01] or \
             [r for r in head_rows if r["ordinal_weight"] > 0]
BEST = min(candidates, key=lambda r: r["score_error"])["ordinal_weight"]
print("confirming ordinal_weight=%g against 0 with full tuning" % BEST)

## Step 2: confirm with full tuning

The setting that matters: every ModernBERT layer trains, with the settings that reached about 0.79
test accuracy in milestone 2. Two runs: no ordinal term, and the weight chosen above. The first
run's `minutes` tells you what the second will cost.

In [ ]:
full_rows = []
for weight in (0, BEST):
    started = time.time()
    model = EncoderModel(tuning="full", epochs=FULL_EPOCHS, batch_size=32, ordinal_weight=weight, verbose=0)
    model.fit(train)
    full_rows.append(summarize("full, ordinal_weight=%g" % weight, weight, model, started))
    print(json.dumps({k: v for k, v in full_rows[-1].items() if k != "score_error_by_question"}))
table(full_rows)

In [ ]:
by_question(full_rows)

## Reading the results

- **It helps** if `score_error` drops clearly with the ordinal term (say by 10% or more) in the
  full-tuning runs. A difference of a point or two in accuracy is within run-to-run noise with one
  seed; to be sure of a small effect, repeat step 2 with `seed=1` and `seed=2`.
- **It costs nothing** if `score_accuracy` stays within about a point, and `choice_accuracy` and
  `noul_accuracy` don't move beyond noise. They can move a little: every question type shares the
  head, and early stopping watches the combined loss, so the term changes when training stops.
- **The default** stays at 1.0 if the best weight is near 1. If 2 is clearly better with no
  accuracy cost, the default should go up; if 0.5 gets most of the gain, a lighter default is
  safer; if no weight helps, the term should default to 0.
- **Per question:** if one score question improves while another gets worse in the
  `by_question` tables, different questions want different weights, which is the case for the
  per-question `ordinal_weight` planned in the design.

Report the two tables from step 2, and the step 1 table, back on the milestone 3 pull request.